In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
# Custom Dataset Class
class UnderwaterDataset(Dataset):
    def __init__(self, root_dir, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, 'dataset',"image")
        mask_path = os.path.join(self.root_dir, 'dataset', "masks")

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Convert mask to grayscale (1 channel = binary segmentation)

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        # Replace mask values with remapped values
        mask = remap_mask(mask)

        return image, mask

In [ ]:

dataset_root = os.path.join(path, "dataset")
image_path = os.path.join(path, "dataset", 'images')
mask_path = os.path.join(path, "dataset",'masks')

image_paths = []
mask_paths = []


for img_p in image_path:
  image_paths.append(img_p)

for mask_p in mask_path:
  mask_paths.append(mask_p)

image_transforms = transforms.Compose([
  transforms.ToTensor(),
  transforms.Resize((256, 256)),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


mask_transforms = transforms.Compose([
  transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
  transforms.PILToTensor(),
])


print(f"Total images: {len(image_paths)}")
print(f"Total masks: {len(mask_paths)}")

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

train_images, test_images, train_masks, test_masks = train_test_split(
  image_paths[0:42], mask_paths, test_size=0.2, random_state=42
)

# Create Dataset objects
train_dataset = UnderwaterDataset(train_images, train_masks, transform=image_transforms, target_transform=mask_transforms)
test_dataset = UnderwaterDataset(test_images, test_masks, transform=image_transforms, target_transform=mask_transforms)

# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
def denormalize(img):
  mean = np.array([0.485, 0.456, 0.406])
  std = np.array([0.229, 0.224, 0.225])
  img = img.permute(1, 2, 0).numpy()  # CHW -> HWC
  img = img * std + mean
  img = np.clip(img, 0, 1)
  return img

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    image, mask = train_dataset[i]

    # Display image (denormalize first)
    axes[0, i].imshow(denormalize(image))
    axes[0, i].set_title(f"Undderwater image {i+1}")
    axes[0, i].axis("off")

    # Display mask
    axes[1, i].imshow(mask.squeeze(), cmap="gray")
    axes[1, i].set_title(f" Mask {i+1}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  #
).to(device)

In [ ]:
import torch.optim as optim
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
  model.train()
  total_loss = 0

  for images, masks in tqdm(dataloader):
    # Move data to device
    images, masks = images.to(device), masks.to(device).float()

    outputs = model(images)
    loss = criterion(outputs, masks)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
def validate(model, dataloader, criterion, device):
  model.eval()
  total_loss = 0

  with torch.no_grad():
    for images, masks in dataloader:
      images, masks = images.to(device), masks.to(device).float()
      outputs = model(images)
      loss = criterion(outputs, masks)

      total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
import torch
from torch import nn
# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

# Function to denormalize images
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Set model to evaluation mode
model.eval()

# Get some test samples
test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))  # Forward pass

    pred_mask = (pred_mask >= 0.5).cpu().squeeze().numpy()  # Convert to binary mask

    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original Image (Denormalized)
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Ground Truth Mask
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    # Predicted Mask
    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()